# Lecture 02 — Data Engineering

PGR215 Data Collection and Analysis — Kristiania University College

## 1. Hva er Data Engineering?

**Data Engineering** handler om å designe, bygge og vedlikeholde systemer som samler inn, lagrer og klargjør data for analyse.

Data engineers bygger infrastrukturen som gjør det mulig for data scientists og analytikere å jobbe effektivt.

### Data-profesjoner

| Rolle | Fokus | Verktøy |
|-------|-------|--------|
| **Data Engineer** | Bygger pipelines og infrastruktur | Python, SQL, Spark, Airflow |
| **Data Analyst** | Analyserer data, lager rapporter | SQL, Excel, Tableau, Power BI |
| **Data Scientist** | ML-modeller, statistikk, prediksjon | Python, R, TensorFlow, scikit-learn |
| **BI Analyst** | Business intelligence, dashboards | Power BI, Tableau, Looker |

In [ ]:
import pandas as pd
import json
import os

# Data professionals og deres daglige oppgaver
roller = pd.DataFrame({
    'Rolle': ['Data Engineer', 'Data Analyst', 'Data Scientist', 'BI Analyst'],
    'Hovedoppgave': [
        'Bygg ETL-pipelines',
        'Analyser data og lag rapporter',
        'Bygg ML-modeller',
        'Lag dashboards for ledelsen'
    ],
    'Viktigste_skill': ['Python + SQL + Cloud', 'SQL + Excel', 'Python + Statistikk', 'SQL + Visualisering'],
    'Gjennomsnittlonn_NOK': [750_000, 600_000, 800_000, 650_000]
})
display(roller)

## 2. Data Pipeline — De 5 stegene

En data pipeline er et sett med prosesser som flytter data fra kilde til mål:

```
Collection → Ingestion → Preparation → Computation → Presentation
```

1. **Collection**: Samle rådata fra ulike kilder
2. **Ingestion**: Overføre data inn i systemet (batch/streaming)
3. **Preparation**: Rense, validere, transformere
4. **Computation**: Beregne KPI-er, aggregere, analysere
5. **Presentation**: Visualisere resultater (dashboards, rapporter)

In [ ]:
# Demonstrer en komplett mini-pipeline
print("=== STEG 1: COLLECTION ===")
# Les fra flere kilder
df_sales = pd.read_csv('../04_data/sales_data.csv')
print(f"Sales: {df_sales.shape}")

df_products = pd.read_json('../04_data/product_data.json')
print(f"Products: {df_products.shape}")

print("\n=== STEG 2: INGESTION ===")
print(f"Totalt {len(df_sales) + len(df_products)} rader lastet inn")

print("\n=== STEG 3: PREPARATION ===")
print(f"Manglende verdier i sales: {df_sales.isnull().sum().sum()}")
df_sales_clean = df_sales.dropna()
print(f"Etter rensing: {len(df_sales_clean)} rader")

print("\n=== STEG 4: COMPUTATION ===")
if 'Total' in df_sales_clean.columns:
    total = df_sales_clean['Total'].sum()
    avg = df_sales_clean['Total'].mean()
    print(f"Total salg: {total:,.2f}")
    print(f"Gjennomsnitt per ordre: {avg:,.2f}")
else:
    print("Kolonner:", df_sales_clean.columns.tolist())

print("\n=== STEG 5: PRESENTATION ===")
print("Data er klar for visualisering i dashboards!")

## 3. ETL vs. ELT

### ETL (Extract, Transform, Load)
Data transformeres **før** den lastes inn i målsystemet.

```
Kilder → EXTRACT → TRANSFORM (staging) → LOAD → Data Warehouse
```

### ELT (Extract, Load, Transform)
Data lastes **først**, deretter transformeres i målsystemet.

```
Kilder → EXTRACT → LOAD → Data Warehouse → TRANSFORM
```

| Aspekt | ETL | ELT |
|--------|-----|-----|
| **Transform** | Før lasting | Etter lasting |
| **Hastighet** | Tregere (transform først) | Raskere innlasting |
| **Fleksibilitet** | Mindre (forhåndsdefinert) | Mer (transform on-demand) |
| **Lagring** | Data Warehouse | Data Lake |
| **Best for** | Strukturert data | Alle datatyper |

In [ ]:
# ETL-eksempel
print("=== ETL: Transform FØR Load ===")

# Extract
raw = pd.DataFrame({
    'name': ['  anna ', 'ERIK', 'sara  ', None],
    'salary': ['500000', '650000', 'unknown', '700000'],
    'date': ['2024-01-15', '2024/02/20', '15.03.2024', '2024-04-10']
})
print("Rå data:")
display(raw)

# Transform (i staging area)
df_etl = raw.copy()
df_etl['name'] = df_etl['name'].str.strip().str.title()
df_etl['salary'] = pd.to_numeric(df_etl['salary'], errors='coerce')
df_etl = df_etl.dropna(subset=['name'])
print("\nEtter Transform:")
display(df_etl)

# Load
print("\n→ Klar til å lastes inn i Data Warehouse!")

print("\n" + "="*50)
print("=== ELT: Load FØR Transform ===")
print("1. Load: Dump alt rått inn i Data Lake")
print("2. Transform: Kjør SQL/Spark queries for å rense on-demand")
print("→ Mer fleksibelt, men krever kraftigere infrastruktur")

## 4. Bronze / Silver / Gold Layers (Medallion Architecture)

Data kvalitetslag i en moderne data-plattform:

| Layer | Beskrivelse | Eksempel |
|-------|------------|----------|
| **Bronze** | Rå, ufiltrert data direkte fra kildene | API-dump, rå CSV-filer |
| **Silver** | Renset, filtrert og standardisert | Fjernet duplikater, fikset typer |
| **Gold** | Forretningsklart, aggregert | KPI-er, rapporter, ML-features |

In [ ]:
# Demonstrer Bronze → Silver → Gold
print("=== BRONZE LAYER (rå data) ===")
bronze = pd.DataFrame({
    'kunde_id': [1, 2, 2, 3, 4, None],
    'produkt': ['Laptop', 'mus', 'Mus', 'TASTATUR', 'laptop', 'Skjerm'],
    'belop': [12000, 299, 299, 890, 11500, 3500],
    'dato': ['2024-01-10', '2024-01-11', '2024-01-11', '2024-01-12', '2024-01-12', '2024-01-13']
})
display(bronze)
print(f"Rader: {len(bronze)}, Nulls: {bronze.isnull().sum().sum()}, Duplikater: {bronze.duplicated().sum()}")

print("\n=== SILVER LAYER (renset) ===")
silver = bronze.copy()
silver = silver.dropna(subset=['kunde_id'])           # Fjern rader uten kunde
silver['kunde_id'] = silver['kunde_id'].astype(int)
silver['produkt'] = silver['produkt'].str.strip().str.title()  # Standardiser
silver = silver.drop_duplicates()                        # Fjern duplikater
silver['dato'] = pd.to_datetime(silver['dato'])
display(silver)
print(f"Rader: {len(silver)}, Nulls: {silver.isnull().sum().sum()}")

print("\n=== GOLD LAYER (aggregert) ===")
gold = silver.groupby('produkt').agg(
    antall_salg=('belop', 'count'),
    total_belop=('belop', 'sum'),
    snitt_belop=('belop', 'mean')
).round(0)
display(gold)
print("\n→ Klar for dashboard og rapporter!")

## 5. Data Extraction

Første steg i ETL — hente data fra ulike kilder:

- **Databaser**: SQL queries (SELECT, JOIN)
- **Filer**: CSV, JSON, Excel, XML, Parquet
- **API-er**: REST/HTTP requests
- **Web Scraping**: BeautifulSoup, Selenium
- **Streams**: Kafka, real-time feeds

In [ ]:
# Extract fra ulike filformater
print("=== Extract fra CSV ===")
df_csv = pd.read_csv('../04_data/sales_data.csv')
print(f"CSV: {df_csv.shape[0]} rader, {df_csv.shape[1]} kolonner")
display(df_csv.head(3))

print("\n=== Extract fra JSON ===")
df_json = pd.read_json('../04_data/product_data.json')
print(f"JSON: {df_json.shape[0]} rader, {df_json.shape[1]} kolonner")
display(df_json.head(3))

print("\n=== Extract fra Excel ===")
try:
    df_xlsx = pd.read_excel('../04_data/customer_data.xlsx')
    print(f"Excel: {df_xlsx.shape[0]} rader, {df_xlsx.shape[1]} kolonner")
    display(df_xlsx.head(3))
except Exception as e:
    print(f"Feil: {e}")

print("\n=== Oversikt over 04_data/ ===")
for f in sorted(os.listdir('../04_data/')):
    size = os.path.getsize(f'../04_data/{f}')
    print(f"  {f:<35} {size:>8,} bytes")

## 6. Data Transformation

Transformasjon er det viktigste steget — gjør rå data brukbar:

| Teknikk | Beskrivelse |
|---------|------------|
| **Cleaning** | Fiks feil, manglende verdier |
| **Filtering** | Velg bare det du trenger |
| **Joining** | Slå sammen datasett |
| **Normalizing** | Konverter til felles enheter |
| **Aggregating** | Summer, tell, gjennomsnitt |
| **Feature Engineering** | Lag nye kolonner for analyse/ML |
| **Anonymizing** | Fjern personlig info (GDPR) |

In [ ]:
# Demonstrer transformasjonsteknikker
df = pd.DataFrame({
    'navn': ['Anna Berg', 'erik hansen', '  Sara Olsen  ', 'OLE LIE', None],
    'epost': ['anna@test.no', 'ERIK@test.NO', 'sara@test.no', None, 'ole@test.no'],
    'lonn': [550000, 0, 620000, 480000, 700000],
    'avdeling': ['IT', 'it', 'HR', 'IT', 'hr'],
    'ansatt_dato': ['2020-03-15', '2019/07/01', '15.01.2021', '2022-11-10', '2018-05-20']
})
print("Rå data:")
display(df)

# 1. Cleaning: Standardiser tekst
df['navn'] = df['navn'].str.strip().str.title()
df['epost'] = df['epost'].str.lower()
df['avdeling'] = df['avdeling'].str.upper()

# 2. Filtering: Fjern ugyldige rader
df = df.dropna(subset=['navn'])
df = df[df['lonn'] > 0]

# 3. Feature Engineering: Lag nye kolonner
df['fornavn'] = df['navn'].str.split().str[0]
df['etternavn'] = df['navn'].str.split().str[-1]

print("\nEtter transformasjon:")
display(df)

## 7. Data Loading

Siste steg — lagre transformert data til målsystem:

- **Database**: INSERT INTO, COPY
- **Data Warehouse**: Bulk load
- **Filer**: CSV, Parquet, JSON
- **Data Lake**: Raw dump

### Loading strategier:
- **Full load**: Erstatt alt hver gang
- **Incremental load**: Bare legg til nye/endrede rader
- **Upsert**: Oppdater eksisterende, legg til nye

In [ ]:
# Demonstrer loading-strategier
import datetime

df_existing = pd.DataFrame({
    'id': [1, 2, 3],
    'navn': ['Anna', 'Erik', 'Sara'],
    'status': ['aktiv', 'aktiv', 'aktiv']
})

df_new = pd.DataFrame({
    'id': [3, 4],
    'navn': ['Sara', 'Ole'],
    'status': ['inaktiv', 'aktiv']
})

print("=== Eksisterende data ===")
display(df_existing)
print("\n=== Ny data ===")
display(df_new)

# FULL LOAD: Erstatt alt
print("\n=== Full Load (erstatt alt) ===")
df_full = df_new.copy()
display(df_full)

# INCREMENTAL: Bare nye rader
print("\n=== Incremental Load (bare nye) ===")
existing_ids = set(df_existing['id'])
df_incremental = pd.concat([df_existing, df_new[~df_new['id'].isin(existing_ids)]])
display(df_incremental)

# UPSERT: Oppdater + legg til
print("\n=== Upsert (oppdater + nye) ===")
df_upsert = pd.concat([df_existing, df_new]).drop_duplicates(subset=['id'], keep='last').sort_values('id')
display(df_upsert)

## 8. Data Staging (Bronze/Silver/Gold)

Mellom transformasjoner lagres data midlertidig i et **staging area**:

- **Bronze**: Rå data, direkte fra kilde (uendret)
- **Silver**: Renset, filtrert, standardisert
- **Gold**: Forretningsklart, aggregert, KPI-er

Dette kalles **Medallion Architecture** og brukes av Databricks og andre moderne plattformer.

## 9. ETL vs. ELT — Når bruke hva?

| Scenario | Anbefaling | Grunn |
|----------|-----------|-------|
| Strukturert data → Warehouse | **ETL** | Transform før lagring er effektivt |
| Ukjent/varierende data | **ELT** | Lagre alt, analyser senere |
| Real-time behov | **ELT** | Raskere innlasting |
| GDPR/personvern | **ETL** | Anonymiser FØR lagring |
| Machine Learning | **ELT** | Data scientists trenger rå data |

## 10. Data Quality

Datakvalitet måles langs flere dimensjoner:

| Dimensjon | Beskrivelse |
|-----------|------------|
| **Completeness** | Er all nødvendig data til stede? |
| **Accuracy** | Er verdiene korrekte? |
| **Consistency** | Er data konsistent på tvers av kilder? |
| **Timeliness** | Er data oppdatert nok? |
| **Validity** | Følger data forventet format? |

In [ ]:
# Datakvalitetssjekk
df_check = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'email': ['anna@test.no', 'ugyldig', None, 'sara@test.no', 'ole@test.no'],
    'alder': [25, 150, 30, -5, 42],
    'lonn': [500000, 600000, None, 550000, 480000]
})

print("=== Datakvalitetsrapport ===")
print(f"\nCompleteness (manglende verdier):")
for col in df_check.columns:
    missing = df_check[col].isnull().sum()
    pct = (1 - missing / len(df_check)) * 100
    print(f"  {col}: {pct:.0f}% komplett ({missing} manglende)")

print(f"\nValidity:")
# Sjekk email-format
valid_email = df_check['email'].str.contains('@.*\\.\\w+', na=False).sum()
print(f"  Gyldige e-poster: {valid_email}/{len(df_check)}")

# Sjekk aldersintervall
valid_age = ((df_check['alder'] >= 0) & (df_check['alder'] <= 120)).sum()
print(f"  Gyldig alder (0-120): {valid_age}/{len(df_check)}")

print(f"\nAccuracy:")
print(f"  Alder=150 og alder=-5 er åpenbart feil!")

## Oppsummering

**Nøkkelkonsepter fra Lecture 02:**

1. Data Engineering = bygge pipelines og infrastruktur
2. Roller: Data Engineer, Analyst, Scientist, BI
3. Pipeline: Collection → Ingestion → Preparation → Computation → Presentation
4. ETL (transform først) vs. ELT (load først)
5. Bronze/Silver/Gold (Medallion Architecture)
6. Extract fra: CSV, JSON, Excel, API, web scraping
7. Transform: cleaning, filtering, joining, aggregating
8. Load: full load, incremental, upsert
9. Data Quality: completeness, accuracy, consistency, timeliness, validity